# 05 · Voltage-to-Duty Inverse Mapping

**Purpose.** Build and validate an inverse model `Duty = f(Voltage)` for closed-loop fan control, compare candidate model forms (polynomial vs. exponential vs. log-linear), quantify prediction error, and demonstrate the chosen model on synthetic voltage profiles.

**Story:** average the raw voltage/duty readings → fit candidate models → select and validate the exponential model → apply it to example voltage traces.

Many exploratory fitting attempts (linear interpolation, spline fits, piecewise exponential fits, an earlier unaveraged error pipeline) live in the Experiments notebook — they informed the final model choice below but are not part of the production pipeline.

## Average raw voltage readings by duty cycle

Loads the raw Excel sheet, cleans column names, groups by duty cycle, and extracts the averaged Duty/Voltage arrays used by every fit below.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load Excel file and correct sheet
file_path = "z=2,r=0.xlsx"
df = pd.read_excel(file_path, sheet_name='21')

# Clean up column names
df.columns = df.columns.str.strip()

# Drop rows with missing values in relevant columns
df = df.dropna(subset=['Duty', 'Voltage'])

# Group by duty cycle and compute mean voltage
grouped = df.groupby('Duty')['Voltage'].mean().reset_index()
grouped = grouped.sort_values(by='Duty')  # Ensure sorting

# ✅ Extract arrays
duties = grouped['Duty'].values          # Duty cycle array
voltages = grouped['Voltage'].values     # Average voltage array

# Print to confirm
print("Duty Cycle Array:", duties)
print("Voltage Array:", voltages)

# Plot
xticks = np.arange(0, 105, 2)
yticks = np.arange(0, 3.6, 0.2)

plt.figure(figsize=(12, 6))
plt.plot(duties, voltages, marker='o', linestyle='-', color='purple')
plt.xticks(xticks)
plt.yticks(yticks)
plt.xlabel('Duty Cycle (%)')
plt.ylabel('Average Voltage (V)')
plt.title('Duty Cycle vs Average Voltage (z = 2, r = 0)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Candidate model 1 — 3rd-degree polynomial fit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Voltage and Duty arrays from your data
voltages = np.array([1.00235314, 0.82740628, 0.81370782, 0.81502292, 0.81771553,
                     0.8102661, 0.80432675, 0.79196567, 0.77636817, 0.76105513,
                     0.74830033, 0.73875982, 0.72980107, 0.72330292, 0.71230216,
                     0.71144245, 0.70705632, 0.70255353, 0.70394202, 0.70389736])
duties = np.array([5, 10, 15, 20, 25, 30, 35, 40, 45, 50,
                   55, 60, 65, 70, 75, 80, 85, 90, 95, 100])

# Fit a 3rd-degree polynomial: Duty = f(Voltage)
coeffs = np.polyfit(voltages, duties, deg=3)
poly = np.poly1d(coeffs)

# Print the regression equation
print("Polynomial Coefficients (highest degree first):", coeffs)
print(f"\nRegression equation:\nDuty = {coeffs[0]:.2f}V³ + {coeffs[1]:.2f}V² + {coeffs[2]:.2f}V + {coeffs[3]:.2f}")

# Generate smooth curve for plotting
v_fit = np.linspace(min(voltages), max(voltages), 500)
d_fit = poly(v_fit)

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(voltages, duties, 'ro', label='Actual Data')
plt.plot(v_fit, d_fit, 'b-', label='Fitted Curve')
plt.xlabel('Voltage (V)')
plt.ylabel('Duty Cycle (%)')
plt.title('Duty Cycle vs Voltage (z=2, r=0)')

# Fine ticks
plt.xticks(np.arange(0.65, 1.05, 0.02))   # finer voltage ticks
plt.yticks(np.arange(0, 100, 3))          # finer duty cycle ticks

plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

## Diagnostic — raw (unaveraged) Duty vs. Voltage scatter

Sanity check on measurement noise before committing to an averaged fit.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the Excel file
file_path = "z=2,r=0.xlsx"
excel_data = pd.ExcelFile(file_path)
df = excel_data.parse('21')  # Use correct sheet

# Clean column names
df.columns = df.columns.str.strip()

# Drop rows with missing values in 'Duty' or 'Voltage'
df = df.dropna(subset=['Duty', 'Voltage'])

# Plot: Each voltage reading vs its corresponding duty cycle
plt.figure(figsize=(10, 5))
plt.scatter(df['Duty'], df['Voltage'], color='purple', s=25, label='Raw Voltage Readings')

plt.xlabel("Duty Cycle (%)")
plt.ylabel("Voltage (V)")
plt.title("Raw Duty Cycle vs Voltage (No Averaging)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.legend()
plt.show()

## Candidate model 2 — tuned exponential fit (chosen model)

`Duty = A · exp(-B · V) + C`, evaluated with RMSE and R². This exponential form was selected as the production inverse-mapping model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score

# Your data
voltages = np.array([1.002, 0.827, 0.814, 0.815, 0.818,
                     0.810, 0.804, 0.792, 0.776, 0.761,
                     0.748, 0.739, 0.730, 0.723, 0.712,
                     0.711, 0.707, 0.703, 0.704, 0.704])
duties = np.array([5, 10, 15, 20, 25,
                   30, 35, 40, 45, 50,
                   55, 60, 65, 70, 75,
                   80, 85, 90, 95, 100])

# Define model: Exponential decay + offset
def exp_model(v, A, B, C):
    return A * np.exp(-B * v) + C

# Initial guess: [Amplitude, Decay rate, Offset]
initial_guess = [300, 10, 0]

# Fit the curve
params, _ = curve_fit(exp_model, voltages, duties, p0=initial_guess)
A_fit, B_fit, C_fit = params

# Print the fitted equation
print(f"Fitted Exponential Equation:\nDuty = {A_fit:.2f} * exp(-{B_fit:.2f} * V) + {C_fit:.2f}")

# Plotting
v_fit = np.linspace(min(voltages), max(voltages), 200)
d_fit = exp_model(v_fit, *params)

plt.figure(figsize=(10, 5))
plt.plot(voltages, duties, 'ro', label='Data')
plt.plot(v_fit, d_fit, 'b-', label='Fitted Curve')
plt.xlabel('Voltage (V)')
plt.ylabel('Duty Cycle (%)')
plt.title('Tuned Exponential Fit')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# ERROR METRICS

duty_pred = exp_model(voltages, *params)
rmse = np.sqrt(mean_squared_error(duties, duty_pred))
r2 = r2_score(duties, duty_pred)

print(f"\nModel Evaluation:")
print(f"RMSE = {rmse:.4f}")
print(f"R²    = {r2:.4f}")

## Apply the chosen exponential model and quantify error

Re-loads and re-averages the raw data, applies the fixed exponential coefficients from the previous cell, computes prediction error, and saves `error_analysis_avg_voltage.csv` — the dataset all subsequent comparison cells read from.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the original raw data
df_raw = pd.read_excel("z=2,r=0.xlsx", sheet_name='21')
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.dropna(subset=['Voltage', 'Duty'])

# Group by Duty cycle to get averaged voltage
df_avg = df_raw.groupby('Duty')['Voltage'].mean().reset_index()
df_avg = df_avg.sort_values(by='Duty')

# Define the exponential function (from your earlier fit)
def exp_model(v):
    return 312233.9 * np.exp(-11.56 * v) - 0.65

# Apply model to averaged voltage to predict duty
df_avg['Predicted Duty'] = exp_model(df_avg['Voltage'])
df_avg['Error'] = df_avg['Predicted Duty'] - df_avg['Duty']
df_avg['% Error'] = 100 * df_avg['Error'].abs() / df_avg['Duty']

# Extract values
actual = df_avg['Duty'].values
predicted = df_avg['Predicted Duty'].values
error = df_avg['Error'].values

# Compute error metrics
rmse = np.sqrt(mean_squared_error(actual, predicted))
mae = mean_absolute_error(actual, predicted)
r2 = r2_score(actual, predicted)
max_error = np.max(np.abs(error))

# Print metrics
print(f"Error Metrics (Averaged Voltage):")
print(f"RMSE         = {rmse:.4f}")
print(f"MAE          = {mae:.4f}")
print(f"R² Score     = {r2:.4f}")
print(f"Max Abs Error = {max_error:.4f}")

# Plot error vs voltage
plt.figure(figsize=(10, 5))
plt.plot(df_avg['Voltage'], df_avg['Error'], marker='o', linestyle='-', color='blue', label='Absolute Error')
plt.xlabel('Average Voltage (V)')
plt.ylabel('Error in Duty Cycle (%)')
plt.title('Duty Prediction Error vs Averaged Voltage')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Plot % Error
plt.figure(figsize=(10, 5))
plt.plot(df_avg['Voltage'], df_avg['% Error'], marker='s', linestyle='--', color='green', label='Percentage Error')
plt.xlabel('Average Voltage (V)')
plt.ylabel('Percentage Error (%)')
plt.title('Percentage Error vs Averaged Voltage')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

df_avg.to_csv("error_analysis_avg_voltage.csv", index=False)

## Refit exponential / polynomial / log-linear models on the working voltage range (0.7–1.0 V)

Restricts to the operating voltage range, refits all three candidate model forms, reports error metrics for each, and saves `refitted_model_error_analysis.csv`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# === Load the data ===
df = pd.read_csv("error_analysis_avg_voltage.csv")
df = df[(df['Voltage'] >= 0.7) & (df['Voltage'] <= 1.0)]  # Use only trimmed range

V = df['Voltage'].values
D_actual = df['Duty'].values

# === 1. Define the models ===
def exp_model(x, a, b, c):
    return a * np.exp(b * x) + c

# Fit models
params_exp, _ = curve_fit(exp_model, V, D_actual, p0=(100, -5, 0))
poly_coeffs = np.polyfit(V, D_actual, 2)
poly_model = np.poly1d(poly_coeffs)

valid_idx = V > 0
log_coeffs = np.polyfit(np.log(V[valid_idx]), D_actual[valid_idx], 1)
log_model = lambda x: log_coeffs[0] * np.log(x) + log_coeffs[1]

# === 2. Predict using all models ===
df['Exp Duty'] = exp_model(V, *params_exp)
df['Poly Duty'] = poly_model(V)
df['Log Duty'] = log_model(V)

# === 3. Calculate error metrics and deltas ===
models = ['Exp Duty', 'Poly Duty', 'Log Duty']
for model in models:
    df[f'{model} Error'] = df[model] - df['Duty']
    df[f'{model} % Error'] = 100 * (df[f'{model} Error'] / df['Duty']).abs()

# === 4. Export the new CSV ===
df.to_csv("refitted_model_error_analysis.csv", index=False)

# === 5. Print summary stats ===
for model in models:
    predicted = df[model].values
    error = df[f'{model} Error'].values
    print(f"\n--- {model} ---")
    print(f"RMSE         = {np.sqrt(mean_squared_error(D_actual, predicted)):.4f}")
    print(f"MAE          = {mean_absolute_error(D_actual, predicted):.4f}")
    print(f"R² Score     = {r2_score(D_actual, predicted):.4f}")
    print(f"Max Abs Error = {np.max(np.abs(error)):.4f}")

## Compare original vs. refitted exponential coefficients

Side-by-side RMSE / MAE / R² / max-error comparison between the original full-range exponential fit and the refitted model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# === Load CSV ===
df = pd.read_csv("error_analysis_avg_voltage.csv")
V = df["Voltage"]
D_actual = df["Duty"]

# === Define both models ===
def model_a(V):
    return 312233.9 * np.exp(-11.56 * V) - 0.65  # Your original

def model_b(V):
    return 15978.86 * np.exp(-6.90 * V) - 34.73  # Refitted

# === Predict ===
df["Model A Duty"] = model_a(V)
df["Model B Duty"] = model_b(V)

# === Error metrics function ===
def error_metrics(actual, predicted):
    return {
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAE": mean_absolute_error(actual, predicted),
        "R2": r2_score(actual, predicted),
        "Max Error": np.max(np.abs(actual - predicted))
    }

# === Compute metrics ===
metrics_a = error_metrics(D_actual, df["Model A Duty"])
metrics_b = error_metrics(D_actual, df["Model B Duty"])

# === Print side-by-side ===
print("Metric         | Model A (Yours) | Model B (Refit)")
print("---------------|----------------|----------------")
for key in metrics_a:
    print(f"{key:<14} | {metrics_a[key]:<14.4f} | {metrics_b[key]:.4f}")

## Apply the chosen model to example (synthetic) voltage profiles

Generates two representative voltage traces — a smooth peaked profile and a stepped plateau profile — converts them to duty cycle using the chosen exponential model, and exports the resulting duty/voltage/time tables.

In [ ]:
import numpy as np
import pandas as pd

# ----------- Graph 1: Smooth Peaks -------------

# Time segments
t1 = np.linspace(0, 10, 50)
t2 = np.linspace(10, 20, 50)
t3 = np.linspace(20, 30, 50)
t4 = np.linspace(30, 40, 50)

# Voltage segments
v1 = np.linspace(0.77, 0.85, len(t1))
v2 = np.linspace(0.85, 0.77, len(t2))
v3 = np.linspace(0.77, 0.85, len(t3))
v4 = np.linspace(0.85, 0.77, len(t4))

# Combine time and voltage
time1 = np.concatenate([t1, t2, t3, t4])
volt1 = np.concatenate([v1, v2, v3, v4])

# Calculate duty from exponential equation
def calculate_duty(v):
    return 312233.9 * np.exp(-11.56 * v) - 0.65

duty1 = calculate_duty(volt1)

# ----------- Graph 2: Stepped Plateau -------------

voltage_steps = [0.77, 0.78, 0.79, 0.80, 0.79, 0.78, 0.77]
durations = [10] * len(voltage_steps)

# Generate time and voltage values
time2 = []
volt2 = []

current_time = 0
for v, dur in zip(voltage_steps, durations):
    t_segment = np.linspace(current_time, current_time + dur, 25)  # 25 samples per plateau
    time2.extend(t_segment)
    volt2.extend([v] * len(t_segment))
    current_time += dur

time2 = np.array(time2)
volt2 = np.array(volt2)

duty2 = calculate_duty(volt2)

# ----------- Output as DataFrames -------------

df1 = pd.DataFrame({'Time (s)': time1, 'Voltage (V)': volt1, 'Duty (%)': duty1})
df2 = pd.DataFrame({'Time (s)': time2, 'Voltage (V)': volt2, 'Duty (%)': duty2})

# Print preview
print("Smooth Voltage Graph (Graph 1):")
print(df1.head(10), '\n')

print("Stepped Voltage Graph (Graph 2):")
print(df2.head(10), '\n')

# Optional: Save to CSV
df1.to_csv("graph1_smooth_voltage_duty.csv", index=False)
df2.to_csv("graph2_stepped_voltage_duty.csv", index=False)